<a href="https://colab.research.google.com/github/droyktton/ICNPG/blob/master/Clases/Colaboratory/nvidiaHPCSDK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%html
<marquee style='width: 100%; color: blue;'><b>ICNPG2021 en Google Colaboratory-Instituto Balseiro </b></marquee>


# NVIDIA HPC SDK

Bajamos el sdk primero, el ultimo (lleva un minuto)

In [ ]:
!wget https://developer.download.nvidia.com/hpc-sdk/21.3/nvhpc_2021_213_Linux_x86_64_cuda_11.2.tar.gz

lo descompactamos (toma un par de minutos)

In [ ]:
!tar xpzf nvhpc_2021_213_Linux_x86_64_cuda_11.2.tar.gz

y para ahorrar espacio en disco vamos borrando los tar.gz

In [ ]:
!rm nvhpc_2021_213_Linux_x86_64_cuda_11.2.tar.gz

y largamos el script de instalacion. Apretar en el output para ver la barra donde podemos completar. Elegimos /content/opt/nvidia/hpc_sdk como carpeta de instalacion, para que este cerca de la eleccion usual. Lleva un par de minutos...


In [ ]:
!nvhpc_2021_213_Linux_x86_64_cuda_11.2/install

Como nos vamos quedando sin espacio limitado de disco que nos dan en la version gratuita, podemos ya borrar la carpeta de instaladores

In [ ]:
!rm -r /content/nvhpc_2021_213_Linux_x86_64_cuda_11.2


Para chequear que ya tenemos el nvidia hpc sdk listo para usar podemos chequiar la presencia de nvfortran, que no esta en el toolkit normal.

In [ ]:
!/content/opt/nvidia/hpc_sdk/Linux_x86_64/21.3/compilers/bin/nvfortran --version

Chequiemos la placa que nos toco y carguemos los plugins para poder escribir algo distinto a python.

In [ ]:
!nvidia-smi

lindas GPUs!!. Ahora, para poder correr Cuda C/C++ instalamos un plugin

In [ ]:
!pip install git+git://github.com/droyktton/nvcc4jupyter.git


y luego:

In [ ]:
%load_ext nvcc_plugin


Listo!, con eso ya podemos correr codigo CUDA C/C++ o Fortran en el notebook Jupyter.


Si uno quiere trabajar con su drive ejecutar esto para montarlo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# CUDA Fortran

Antes de cargar y modificar el codigo principal es importante subir todos los archivos auxiliares que se usan y que no se van a editar y ponerlos en /content/ . Eso se hace en la seccion de la izquierda, colocandose en la carpeta /content/ primero y luego apretando el botoncito de upload (flecha para arriba).

Ahora si, cargamos un codigo que podemos ir modificando. En este caso el clasico problema saxpy.

In [ ]:
%%cuda --name test.cu
module mathOps
contains
  attributes(global) subroutine saxpy(x, y, a)
    implicit none
    real :: x(:), y(:)
    real, value :: a
    integer :: i, n
    n = size(x)
    i = blockDim%x * (blockIdx%x - 1) + threadIdx%x
    if (i <= n) y(i) = y(i) + a*x(i)
  end subroutine saxpy 
end module mathOps

program testSaxpy
  use mathOps
  use cudafor
  implicit none
  integer, parameter :: N = 40000
  real :: x(N), y(N), a
  real, device :: x_d(N), y_d(N)
  type(dim3) :: grid, tBlock

  tBlock = dim3(256,1,1)
  grid = dim3(ceiling(real(N)/tBlock%x),1,1)

  x = 1.0; y = 2.0; a = 2.0
  x_d = x
  y_d = y
  call saxpy<<<grid, tBlock>>>(x_d, y_d, a)
  y = y_d
  write(*,*) 'Max error: ', maxval(abs(y-4.0))
end program testSaxpy

In [ ]:
!cp /content/src/test.cu /content/src/test.f90; /content/opt/nvidia/hpc_sdk/Linux_x86_64/21.3/compilers/bin/nvfortran  /content/src/test.f90  -cuda  -L/content/lib

notar que tuvimos que renombrar a test.cu como test.f90. Eso pasa porque el plugin acepta solo .cu mientras que el compilador de fortran espera extensiones .f, .f90 o .f95

Ya podemos correr el codigo

In [ ]:
!./a.out

 Max error:     0.000000    


Parece andar!! 

# OPENACC


El ejemplo del problema de Jacobi, que vimos en clase, acelerado con openacc.

In [ ]:
%%cuda --name test.cu
/*
 *  Copyright 2012 NVIDIA Corporation
 *
 *  Licensed under the Apache License, Version 2.0 (the "License");
 *  you may not use this file except in compliance with the License.
 *  You may obtain a copy of the License at
 *
 *      http://www.apache.org/licenses/LICENSE-2.0
 *
 *  Unless required by applicable law or agreed to in writing, software
 *  distributed under the License is distributed on an "AS IS" BASIS,
 *  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 *  See the License for the specific language governing permissions and
 *  limitations under the License.
 */

#include <math.h>
#include <string.h>
//#include "timer.h"
#include <omp.h>
#include <stdio.h>
#include <openacc.h>

#ifndef NN
#define NN 4096
#endif 

#ifndef NM
#define NM 4096
#endif

double A[NN][NM];
double Anew[NN][NM];

int main(int argc, char** argv)
{
    const int n = NN;
    const int m = NM;
    const int iter_max = 1000;
    
    const double tol = 1.0e-6;
    double error     = 1.0;
    
    memset(A, 0, n * m * sizeof(double));
    memset(Anew, 0, n * m * sizeof(double));
        
    for (int j = 0; j < n; j++)
    {
        A[j][0]    = 1.0;
        Anew[j][0] = 1.0;
    }
    
    printf("Jacobi relaxation Calculation: %d x %d mesh\n", n, m);
    
    #if _OPENACC
    acc_init(acc_device_nvidia);
    #endif

    //StartTimer();
    int iter = 0;
    
#pragma acc data copy(A), create(Anew)
    while ( error > tol && iter < iter_max )
    {
        error = 0.0;

#pragma omp parallel for shared(m, n, Anew, A)
#pragma acc kernels
{
        for( int j = 1; j < n-1; j++)
        {
            for( int i = 1; i < m-1; i++ )
            {
                Anew[j][i] = 0.25 * ( A[j][i+1] + A[j][i-1]
                                    + A[j-1][i] + A[j+1][i]);
                error = fmax( error, fabs(Anew[j][i] - A[j][i]));
            }
        }
}
        
#pragma omp parallel for shared(m, n, Anew, A)
#pragma acc kernels
{
        for( int j = 1; j < n-1; j++)
        {
            for( int i = 1; i < m-1; i++ )
            {
                A[j][i] = Anew[j][i];    
            }
        }
}

        if(iter % 100 == 0) printf("%5d, %0.6f\n", iter, error);
        
        iter++;
    }

    //double runtime = GetTimer();
 
    //printf(" total: %f s\n", runtime / 1000);

}


'File written in /content/src/test.cu'

Esta compilacion no tiene en cuenta las directivas openacc y por tanto es serial. Se podria haber usado cualquier compilador de C. Los timers estan comentados, pero se podrian usar cargando /content/timer.h o poniendolo en el drive y corriguendo el path. 

In [ ]:
!nvcc -DNN=1024 -DNM=1024 /content/src/test.cu 

lo ejecutamos

In [ ]:
!time /content/a.out

Ahora compilemos con el compilador pgc++ activando la aceleracion con openACC en placas Nvidia (recordar que openacc nos permite paralelizar para otras plataformas tambien)

In [ ]:
!/content/opt/nvidia/hpc_sdk/Linux_x86_64/21.3/compilers/bin/pgc++ -DNN=1024 -DNM=1024 -acc -ta=nvidia -Minfo=accel /content/src/test.cu 

parece que paralelizo algo!. Corramos la version acelerada ahora

In [ ]:
!time /content/a.out